# FQ-SSIUU Full Experiment Suite Runner

## IMPORTANT: Run on SINGLE T4 GPU

This notebook orchestrates the execution of the full FQ-SSIUU experiment suite (E0b through E7).
- **DRY_RUN functionality**: Set `DRY_RUN = True` in the configuration cell to validate the setup without running full experiments.

### Before running:
1. Set accelerator to **GPU T4 x1** (single GPU)
2. Ensure **Internet** is toggled ON
3. Replace `YOUR_HF_TOKEN` with your actual HuggingFace token

## Cell 1: Setup and Configuration

In [ ]:
import os
import subprocess
import sys

# =================================================================
# CONFIGURATION
# =================================================================
HF_TOKEN = "YOUR_HF_TOKEN"
DRY_RUN = True  # Set to False to run actual experiments

if HF_TOKEN == "YOUR_HF_TOKEN":
    print("WARNING: Please set your actual HF_TOKEN")

def run_cmd(cmd, dry_run=False):
    print(f"\n{'='*50}")
    print(f"Executing: {cmd}")
    print(f"{'='*50}\n")
    if dry_run:
        print("[DRY RUN] Skipping execution.")
        return
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
    
    process = subprocess.Popen(cmd, shell=True, env=env)
    process.communicate()
    if process.returncode != 0:
        print(f"Command failed with exit code {process.returncode}")
        sys.exit(process.returncode)

if not os.path.exists("fq-ssiuu"):
    print("Cloning repository...")
    if DRY_RUN:
        print("[DRY RUN] Skipping git clone.")
    else:
        run_cmd("git clone https://github.com/ibn-e-Muhammad/fq-ssiuu.git")
        run_cmd("cd fq-ssiuu && git checkout dev")
else:
    print("Repository already exists.")

# Write HF token
if not DRY_RUN:
    os.makedirs(os.path.expanduser("~/.cache/huggingface"), exist_ok=True)
    with open(os.path.expanduser("~/.cache/huggingface/token"), "w") as f:
        f.write(HF_TOKEN)

# Environment Setup
print("Setting up environment...")
run_cmd("cd fq-ssiuu && bash scripts/setup_env.sh", dry_run=DRY_RUN)


## Cell 2: Execute Experiment Suite

This runs the core sequence based on the GEMINI_FULL_EXPERIMENT_HANDOFF.md plan.

In [ ]:
# The exact commands will be filled according to the pipeline definitions.
experiments = [
    "python scripts/research_gate.py", # Run gate verification
    # Additional commands for E0b, E1, E2, E3, E4, E7 will go here.
    # Placeholder for demonstration:
    "python -m pytest tests/"
]

for exp_cmd in experiments:
    run_cmd(f"cd fq-ssiuu && {exp_cmd}", dry_run=DRY_RUN)
